# RAG Evaluation Harness (local Ollama)
# - Fill in PDF_PATH and EVAL_SET below
# - Uses the same pipeline as app.py and a lightweight LLM-judge for faithfulness/relevancy


In [ ]:
# Optional: install extras if missing
# !pip install pandas langchain langchain-ollama langchain-community


In [5]:
# Ensure compatible LangChain/Ollama versions
# Run once per env, then restart kernel
!pip install -U "langchain-core>=0.3.0" "langchain-community>=0.3.0" "langchain>=0.3.0" "langchain-ollama>=0.2.0"


  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_text_splitters-1.1.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached langgraph_prebuilt-1.0.8-py3-none-any.whl.metadata (5.2 kB)
  Using cached pydantic_core-2.33.2-cp312-cp312-win_amd64.whl.metadata (6.9 kB)
Using cached langchain_community-0.4.1-py3-none-any.whl (2.5 MB)
Using cached pydantic_core-2.33.2-cp312-cp312-win_amd64.whl (2.0 MB)
Using cached langchain_text_splitters-1.1.1-py3-none-any.whl (35 kB)
Using cached langgraph_prebuilt-1.0.8-py3-none-any.whl (35 kB)
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.41.5
    Uninstalling pydantic_core-2.41.5:
      Successfully uninstalled pydantic_core-2.41.5
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.77
    Uninstalling langchain-core-0.3.77:
      Successfully uninstalled langchain-core-0.3.77
  Attempting uninstall: langchain-text-splitters
   

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
instructor 1.9.0 requires jiter<0.11,>=0.6.1, but you have jiter 0.13.0 which is incompatible.
langchain-experimental 0.3.4 requires langchain-community<0.4.0,>=0.3.0, but you have langchain-community 0.4.1 which is incompatible.
langchain-experimental 0.3.4 requires langchain-core<0.4.0,>=0.3.28, but you have langchain-core 1.2.23 which is incompatible.
langchain-huggingface 0.3.1 requires langchain-core<1.0.0,>=0.3.70, but you have langchain-core 1.2.23 which is incompatible.
langchain-openai 1.1.10 requires openai<3.0.0,>=2.20.0, but you have openai 1.109.1 which is incompatible.


In [2]:
# Ensure LangChain is installed (run once per env, then restart kernel)
!pip install -U "langchain>=0.3.0" "langchain-core>=0.3.0" "langchain-community>=0.3.0"


  Using cached langchain-1.2.14-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_core-1.2.23-py3-none-any.whl.metadata (4.4 kB)
  Using cached langgraph-1.1.4-py3-none-any.whl.metadata (7.4 kB)
Using cached langchain-1.2.14-py3-none-any.whl (112 kB)
Using cached langchain_core-1.2.23-py3-none-any.whl (506 kB)
Using cached langgraph-1.1.4-py3-none-any.whl (168 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.16
    Uninstalling langchain-core-1.2.16:
      Successfully uninstalled langchain-core-1.2.16
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.0.9
    Uninstalling langgraph-1.0.9:
      Successfully uninstalled langgraph-1.0.9
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.10
    Uninstalling langchain-1.2.10:
      Successfully uninstalled langchain-1.2.10


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-experimental 0.3.4 requires langchain-community<0.4.0,>=0.3.0, but you have langchain-community 0.4.1 which is incompatible.
langchain-experimental 0.3.4 requires langchain-core<0.4.0,>=0.3.28, but you have langchain-core 1.2.23 which is incompatible.
langchain-huggingface 0.3.1 requires langchain-core<1.0.0,>=0.3.70, but you have langchain-core 1.2.23 which is incompatible.
langchain-openai 1.1.10 requires openai<3.0.0,>=2.20.0, but you have openai 1.109.1 which is incompatible.


In [1]:
import os
import pandas as pd
from langchain_ollama import ChatOllama

from ingest.load_pdf import ingest_pdf
from ingest.embed_chunks import load_vector_db
from rag.retriever import create_retriever
from rag.chain import create_chain
from config import MODEL_NAME, LLM_TEMPERATURE, LLM_MAX_TOKENS


ModuleNotFoundError: No module named 'langchain.prompts'

In [ ]:
# Set your PDF and evaluation questions
PDF_PATH = "data\\Student Trainee AI Models (m_f_d) _ Rheinmetall.pdf"  # point to a local PDF to test
EVAL_SET = [
    {"question": "What is the main topic of the document?", "reference": ""},
    {"question": "Summarize the key findings.", "reference": ""},
]
assert os.path.exists(PDF_PATH), f"PDF not found: {PDF_PATH}"


In [2]:
# Build RAG components once
llm = ChatOllama(model=MODEL_NAME, temperature=LLM_TEMPERATURE, num_predict=LLM_MAX_TOKENS)

raw_docs, _ = ingest_pdf(open(PDF_PATH, "rb"), clear_existing=True)
vector_db = load_vector_db(raw_docs, clear_existing=True)
retriever = create_retriever(vector_db, llm, retrieval_type=None)  # uses config.RETRIEVAL_TYPE
chain = create_chain(retriever, llm)


NameError: name 'ChatOllama' is not defined

In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

judge_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are grading a retrieval-augmented answer."
               "Score 1-5 for how well the answer is grounded in the given context"
               " and correctly answers the question."
               "Return just a single integer 1-5."),
    ("human", "Question: {question}\n\nContext:\n{context}\n\nAnswer:\n{answer}\n\nScore (1-5):")
])
judge_chain = judge_prompt | llm | StrOutputParser()

def score_answer(question, context, answer):
    raw = judge_chain.invoke({"question": question, "context": context, "answer": answer})
    digits = [c for c in raw if c.isdigit()]
    return int(digits[0]) if digits else None


In [ ]:
results = []
for ex in EVAL_SET:
    q = ex["question"]
    docs = retriever.get_relevant_documents(q)
    context_text = "\n\n".join(d.page_content for d in docs)
    answer = chain.invoke(q)
    score = score_answer(q, context_text, answer)
    results.append({
        "question": q,
        "answer": answer,
        "score_1_5": score,
        "context_used_chars": len(context_text),
    })

pd.DataFrame(results)
